**Overview**

This notebook contains functions for computing the parameters of graphs locally equivalent to a complete multipartite graph or a clique-star. It is intended as supplemental material for the paper "Optimizing Preparation of Distance Hereditary Graph States".
Please compile the entire notebook, and then scroll to the bottom for a summary of the most useful functions for a reader. Small working examples of these are included after the summary and may be freely experimented with.

In [1]:
import numpy as np
import sympy
from IPython.core.display import struct

In [2]:
# Recursive function to create a generator object listing intger partitions.
# (This function was written by Google's generative AI)

def generate_partitions(n, max_val=None):
    if max_val is None:
        max_val = n

    if n == 0:
        yield []
        return

    for i in range(min(max_val, n), 0, -1):
        for p in generate_partitions(n - i, i):
            yield [i] + p

In [3]:
def construct_list_of_multipartite_group_sizes(n,ni_min=2,k_min=3,k_max=7):

  # Initialize an empty list to store the group sizes
  group_sizes_list = []

  # Skip those partitions with less than k_min groups or a group size of ni_min.
  for partition in generate_partitions(n):
    k = len(partition)
    skip = False
    if ((k >= k_min) and (k <= k_max)):
      for i in range(k):
        if partition[i] < ni_min:
          skip = True
      if skip == False:
        group_sizes_list.append(partition)

  return group_sizes_list

In [4]:
# Example of some group sizes
#construct_list_of_multipartite_group_sizes(11,2,3,6)

[[7, 2, 2],
 [6, 3, 2],
 [5, 4, 2],
 [5, 3, 3],
 [5, 2, 2, 2],
 [4, 4, 3],
 [4, 3, 2, 2],
 [3, 3, 3, 2],
 [3, 2, 2, 2, 2]]

In [5]:
def create_product_expression(k):

  # Creata list of variables for n1,...nk
  variable_names = []
  for i in range(k):
    temp_var = "n"+str(i+1)
    variable_names.append(temp_var)

  symbol_objects = [sympy.symbols(name) for name in variable_names]

  # DEBUG:
  #print("Variables names:",variable_names)
  #print("SymPy symbols:",symbol_objects)

  # Create an expression to represent the full expansion of (n1+1)*...*(nk+1)
  product_expression = 1
  for i in range(k):
    product_expression = product_expression * (symbol_objects[i] + 1)

  # DEBUG:
  #print("Expression:",product_expression)
  #print("Expansion:",sympy.expand(product_expression))

  return product_expression

In [6]:
def create_odd_summation_expression(k):

  # Create the fully expanded expression
  product_expression = sympy.expand(create_product_expression(k))

  # DEBUG:
  #print("Fully expanded expression:",product_expression)
  #print("Summands:",product_expression.args)

  # Create a new expression summing just the odd terms.
  # Do this by checking each how many different variables are in each term.
  odd_terms = []
  odd_summation_expression = 0

  for term in product_expression.args:
    # DEBUG
    #print(term.free_symbols)
    if len(term.free_symbols) % 2 == 1:
      odd_terms.append(term)
      odd_summation_expression = odd_summation_expression + term

  # DEBUG
  #print("List of odd product terms:",odd_terms)
  #print("Summation of odd product terms:",odd_summation_expression)

  return odd_summation_expression

In [7]:
def create_even_summation_expression(k):

  # Create the fully expanded expression
  product_expression = sympy.expand(create_product_expression(k))

  # DEBUG:
  #print("Fully expanded expression:",product_expression)
  #print("Summands:",product_expression.args)

  # Create a new expression summing just the even terms.
  # Do this by checking each how many different variables are in each term.
  even_terms = []
  even_summation_expression = 0

  for term in product_expression.args:
    # DEBUG
    #print(term.free_symbols)
    if len(term.free_symbols) % 2 == 0:
      even_terms.append(term)
      even_summation_expression = even_summation_expression + term

  # DEBUG
  #print("List of odd product terms:",even_terms)
  #print("Summation of odd product terms:",even_summation_expression)

  return even_summation_expression

In [8]:
def create_summation_product_expression(k):

  # Creata list of variables for n1,...nk
  variable_names = []
  for i in range(k):
    temp_var = "n"+str(i+1)
    variable_names.append(temp_var)

  symbol_objects = [sympy.symbols(name) for name in variable_names]

  # Nest some loops to create the summation of the products.
  temp_sum = 0
  for j in range(k):
    temp_product = 1
    for i in range(k):
      if i != j:
        temp_product = temp_product * (symbol_objects[i] + 1)
    temp_sum = temp_sum + temp_product

  # DEBUG:
  #print("Final expression:",temp_sum)

  return temp_sum

In [9]:
def evaluate_sum_of_odd_products_formula(list_of_group_sizes):

  # Infer the total number of groups vertices from the list.
  # Also infer the total number of vertices.
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)

  # Create the expression, then evaulate it using the list inputs.
  odd_summation_expression = create_odd_summation_expression(k)
  summation_product_expression = create_summation_product_expression(k)
  combined_expression = odd_summation_expression + summation_product_expression
  free_symbol_list = list(combined_expression.free_symbols)

  # DEBUG
  #print("Odd summation:",odd_summation_expression)
  #print("Summation product:",summation_product_expression)
  #print("Combined expression:",combined_expression)
  #print("Free variables:",free_symbol_list)

  list_of_substitution_pairs = []
  for i in range(k):
    list_of_substitution_pairs.append((free_symbol_list[i],list_of_group_sizes[i]))
  evaluated_expression = combined_expression.subs(list_of_substitution_pairs)

  # DEBUG
  #print("Expression after subsitution",evaluated_expression)

  return evaluated_expression

In [10]:
def evaluate_sum_of_even_products_formula(list_of_group_sizes):

  # Infer the total number of groups vertices from the list.
  # Also infer the total number of vertices.
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)

  # Create the expression, then evaulate it using the list inputs.
  even_summation_expression = create_even_summation_expression(k)
  summation_product_expression = create_summation_product_expression(k)
  combined_expression = even_summation_expression + summation_product_expression
  free_symbol_list = list(combined_expression.free_symbols)

  # DEBUG
  #print("Even summation:",even_summation_expression)
  #print("Summation product:",summation_product_expression)
  #print("Combined expression:",combined_expression)
  #print("Free variables:",free_symbol_list)

  list_of_substitution_pairs = []
  for i in range(k):
    list_of_substitution_pairs.append((free_symbol_list[i],list_of_group_sizes[i]))
  evaluated_expression = combined_expression.subs(list_of_substitution_pairs)

  # DEBUG
  #print("Expression after subsitution",evaluated_expression)

  return evaluated_expression

In [46]:
# Example to compute the size of the LC orbit for the complete-multipartite graph with QASST decomposition [n1,...,nk]
#for part in construct_list_of_multipartite_group_sizes(n=27,ni_min=5,k_min=3,k_max=7):
  #print(evaluate_sum_of_even_products_formula(part),part)

In [47]:
# Example to compute the size of the LC Orbit for clique-star with QASST decomposition [n1,...,nk]
#for part in construct_list_of_multipartite_group_sizes(n=27,ni_min=5,k_min=3,k_max=7):
  #print(evaluate_sum_of_odd_products_formula(part),part)

In [13]:
def comparison_formula(k,nj):
  return ((nj-1)*(k-1) + ((nj-2)*(nj-1)/2) - ((k-2)*(k-1)/2))

In [14]:
def sum_minus_one(list_of_group_sizes):
  k = len(list_of_group_sizes)
  sum_total = 0
  for i in range(k):
    sum_total = sum_total + (list_of_group_sizes[i] - 1)
  return sum_total

In [15]:
 def sum_minus_one_exclude_min(list_of_group_sizes):
  k = len(list_of_group_sizes)
  nj = min(list_of_group_sizes)
  j = list_of_group_sizes.index(nj)
  sum_total = 0
  for i in range(k):
    if (i != j):
      sum_total = sum_total + (list_of_group_sizes[i] - 1)
  return sum_total

In [16]:
def minimal_edge_count_complete_multipartite_orbit(list_of_group_sizes):

  # Infer the total number of groups vertices from the list.
  # Also infer the total number of vertices and some derived terms.
  k = len(list_of_group_sizes)
  nj = min(list_of_group_sizes)
  comp_value = comparison_formula(k,nj)

  # Initialize a number of the minimal edge count.
  # Also initalize a parameter for the case of the structure.
  min_edge = 0
  structure_case = 0

  if ((k%2 == 0) and (comp_value >= 0)):
    structure_case = 1
    min_edge = (k*(k-1)/2) + sum_minus_one(list_of_group_sizes)
  elif ((k%2 == 0) and (comp_value < 0)):
    structure_case = 2
    min_edge = nj*(k-1) + (nj*(nj-1)/2) + sum_minus_one_exclude_min(list_of_group_sizes)
  elif (k%2 == 1):
    structure_case = 3
    min_edge = nj*(k-1) + sum_minus_one_exclude_min(list_of_group_sizes)

  # Return the minimal edge count and the structure
  return min_edge, structure_case

In [48]:
def minimal_edge_count_and_structure_complete_multipartite_orbit(list_of_group_sizes):

  # Infer the parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes)
  j = list_of_group_sizes.index(nj)

  # Infer the number of minimal edges and the structure case
  min_edge, structure_case = minimal_edge_count_complete_multipartite_orbit(list_of_group_sizes)

  # Initilize a string to represent the structure case
  #structure_string_list = []
  structure_dict = {}

  if (structure_case == 1):
    #structure_string_list.append("Q0 = c")
    structure_dict["Q0"] = "c"
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"

  elif (structure_case == 2):
    #tructure_string_list.append("Q0 = sc"+str(j+1))
    structure_dict["Q0"] = "sc"+str(j+1)
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"
    #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
    structure_dict["Q"+str(j+1)] = "c"

  elif (structure_case == 3):
    #structure_string_list.append("Q0 = sc"+str(j+1))
    structure_dict["Q0"] = "sc"+str(j+1)
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"
    #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
    structure_dict["Q"+str(j+1)] = "sc"

  return min_edge, structure_dict

In [50]:
# Example of the minimal edge count and LC orbit representative for K2,2,2
print(minimal_edge_count_and_structure_complete_multipartite_orbit([2,2,2]))

(6, {'Q0': 'sc1', 'Q1': 'sc', 'Q2': 'ss', 'Q3': 'ss'})


In [51]:
#for part in construct_list_of_multipartite_group_sizes(n=20,ni_min=4,k_min=3,k_max=7):
  #print(minimal_edge_count_and_structure_complete_multipartite_orbit(part))

In [52]:
#for part in construct_list_of_multipartite_group_sizes(n=27,ni_min=5,k_min=3,k_max=7):
  #print(minimal_edge_count_and_structure_complete_multipartite_orbit(part),part)

In [21]:
def minimal_edge_count_clique_star_orbit(list_of_group_sizes):

  # Infer the total number of groups vertices from the list.
  # Also infer the total number of vertices and some derived terms.
  k = len(list_of_group_sizes)
  nj = min(list_of_group_sizes)
  comp_value = comparison_formula(k,nj)

  # Initialize a number of the minimal edge count.
  # Also initalize a parameter for the case of the structure.
  min_edge = 0
  structure_case = 0

  if (k%2 == 0):
    structure_case = 1
    min_edge = nj*(k-1) + sum_minus_one_exclude_min(list_of_group_sizes)
  elif ((k%2 == 1) and (comp_value >= 0)):
    structure_case = 2
    min_edge = (k*(k-1)/2) + sum_minus_one(list_of_group_sizes)
  elif ((k%2 == 1) and (comp_value < 0)):
    structure_case = 3
    min_edge = nj*(k-1) + (nj*(nj-1)/2) + sum_minus_one_exclude_min(list_of_group_sizes)

  # Return the minimal edge count and the structure
  return min_edge, structure_case

In [53]:
def minimal_edge_count_and_structure_clique_star_orbit(list_of_group_sizes):

  # Infer the parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes)
  j = list_of_group_sizes.index(nj)

  # Infer the number of minimal edges and the structure case
  min_edge, structure_case = minimal_edge_count_clique_star_orbit(list_of_group_sizes)

  # Initilize a string to represent the structure case
  #structure_string_list = []
  structure_dict = {}

  if (structure_case == 1):
    #structure_string_list.append("Q0 = sc"+str(j+1))
    structure_dict["Q0"] = "sc"+str(j+1)
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"
    #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
    structure_dict["Q"+str(j+1)] = "sc"

  elif (structure_case == 2):
    #structure_string_list.append("Q0 = c")
    structure_dict["Q0"] = "c"
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"

  elif (structure_case == 3):
    #structure_string_list.append("Q0 = sc"+str(j+1))
    structure_dict["Q0"] = "sc"+str(j+1)
    for i in range(k):
      #structure_string_list.append("Q"+str(i+1)+" = ss")
      structure_dict["Q"+str(i+1)] = "ss"
    #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
    structure_dict["Q"+str(j+1)] = "c"

  return min_edge, structure_dict

In [54]:
# Example computing a minimal edge count and LC orbit representative of CS5,2,2,2
minimal_edge_count_and_structure_clique_star_orbit([5,2,2,2])

(12, {'Q0': 'sc2', 'Q1': 'ss', 'Q2': 'sc', 'Q3': 'ss', 'Q4': 'ss'})

In [55]:
#for part in construct_list_of_multipartite_group_sizes(n=27,ni_min=5,k_min=3,k_max=7):
  #print(minimal_edge_count_and_structure_clique_star_orbit(part),part)

In [56]:
#for part in construct_list_of_multipartite_group_sizes(12):
  #print(minimal_edge_count_and_structure_clique_star_orbit(part))

In [26]:
def Delta_G1(list_of_group_sizes):
  # Infer certain parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes) # the smallest entry
  j = list_of_group_sizes.index(nj) # index of the smallest entry
  list_of_group_sizes_reduced = list_of_group_sizes.copy()
  nj_temp = list_of_group_sizes_reduced.pop(j)
  nt = min(list_of_group_sizes_reduced) # the second smallest entry
  t = list_of_group_sizes.index(nt) # index in the original list
  nl = max(list_of_group_sizes) # the largest entry
  l = list_of_group_sizes.index(nl) # index of the largest entry
  # Note: it's okay if the index matches for some, we actually won't use it.

  # Compute the value of Delta_G_1
  case_list = [
      (nl+k-2),
      max((k-1+nt),(nl-1+nj)),
      max((nj+k-2),(nl-1+nj))
  ]

  delta_G = min(case_list)
  case_index = case_list.index(delta_G)+1

  return (delta_G,case_index)

In [27]:
def Delta_G2(list_of_group_sizes):
  # Infer certain parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes) # the smallest entry
  j = list_of_group_sizes.index(nj) # index of the smallest entry
  list_of_group_sizes_reduced = list_of_group_sizes.copy()
  nj_temp = list_of_group_sizes_reduced.pop(j)
  nt = min(list_of_group_sizes_reduced) # the second smallest entry
  t = list_of_group_sizes.index(nt) # index in the original list
  nl = max(list_of_group_sizes) # the largest entry
  l = list_of_group_sizes.index(nl) # index of the largest entry
  # Note: it's okay if the index matches for some, we actually won't use it.

  # Compute the value of Delta_G_1
  case_list = [
      (nl+nj+k-3),
      max((k-1),(nl-1+nj)),
      max((nj+nt+k-3),(nl-1+nj))
  ]

  delta_G = min(case_list)
  case_index = case_list.index(delta_G)+1

  return (delta_G,case_index)

In [28]:
def determine_minimal_Delta_G_complete_multipartite(list_of_group_sizes):
  # Infer certain parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes) # the smallest entry
  j = list_of_group_sizes.index(nj) # index of the smallest entry
  list_of_group_sizes_reduced = list_of_group_sizes.copy()
  nj_temp = list_of_group_sizes_reduced.pop(j)
  nt = min(list_of_group_sizes_reduced) # the second smallest entry
  t = list_of_group_sizes.index(nt) # index in the original list
  if (t == j):
    # If t matches j (possible when nt=nj since it chooses the first index),
    # fix this by shifting t by 1; this is okay because of how sizes are ordered.
    t = t + 1

  # Initilize a string to represent the structure case
  #structure_string_list = []
  structure_dict = {}

  if (k%2 == 0):
    min_Delta_G, structure_case = Delta_G1(list_of_group_sizes)

    if (structure_case == 1):
      #structure_string_list.append("Q0 = c")
      structure_dict["Q0"] = "c"
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"

    elif (structure_case == 2):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      #structure_string_list[t+1] = "Q"+str(t+1)+" = c"
      structure_dict["Q"+str(j+1)] = "sc"
      structure_dict["Q"+str(t+1)] = "c"

    elif (structure_case == 3):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
      structure_dict["Q"+str(j+1)] = "c"

  elif (k%2 == 1):
    min_Delta_G, structure_case = Delta_G2(list_of_group_sizes)

    if (structure_case == 1):
      #structure_string_list.append("Q0 = c")
      structure_dict["Q0"] = "c"
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      structure_dict["Q"+str(j+1)] = "sc"

    elif (structure_case == 2):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      structure_dict["Q"+str(j+1)] = "sc"

    elif (structure_case == 3):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
      #structure_string_list[t+1] = "Q"+str(t+1)+" = c"
      structure_dict["Q"+str(j+1)] = "c"
      structure_dict["Q"+str(t+1)] = "c"

  return min_Delta_G, structure_dict

In [58]:
# Example to compute the minimal maximum-vertex-degree of K2,2,2
determine_minimal_Delta_G_complete_multipartite([2,2,2])

(3, {'Q0': 'sc1', 'Q1': 'sc', 'Q2': 'ss', 'Q3': 'ss'})

In [59]:
#for part in construct_list_of_multipartite_group_sizes(n=9,ni_min=2,k_min=3,k_max=7):
  #print(1+determine_minimal_Delta_G_complete_multipartite(part)[0],determine_minimal_Delta_G_complete_multipartite(part)[1],part)

In [30]:
# Note that the cases for the clique-star are identical to those for the
# complete multipartite graph, except reveresed for the parity of k.

def determine_minimal_Delta_G_clique_star(list_of_group_sizes):
  # Infer certain parameters
  k = len(list_of_group_sizes)
  n = sum(list_of_group_sizes)
  nj = min(list_of_group_sizes) # the smallest entry
  j = list_of_group_sizes.index(nj) # index of the smallest entry
  list_of_group_sizes_reduced = list_of_group_sizes.copy()
  nj_temp = list_of_group_sizes_reduced.pop(j)
  nt = min(list_of_group_sizes_reduced) # the second smallest entry
  t = list_of_group_sizes.index(nt) # index in the original list
  if (t == j):
    # If t matches j (possible when nt=nj since it chooses the first index),
    # fix this by shifting t by 1; this is okay because of how sizes are ordered.
    t = t + 1

  # Initilize a string to represent the structure case
  #structure_string_list = []
  structure_dict = {}

  if (k%2 == 1):
    min_Delta_G, structure_case = Delta_G1(list_of_group_sizes)

    if (structure_case == 1):
      #structure_string_list.append("Q0 = c")
      structure_dict["Q0"] = "c"
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"

    elif (structure_case == 2):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      #structure_string_list[t+1] = "Q"+str(t+1)+" = c"
      structure_dict["Q"+str(j+1)] = "sc"
      structure_dict["Q"+str(t+1)] = "c"

    elif (structure_case == 3):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
      structure_dict["Q"+str(j+1)] = "c"

  elif (k%2 == 0):
    min_Delta_G, structure_case = Delta_G2(list_of_group_sizes)

    if (structure_case == 1):
      #structure_string_list.append("Q0 = c")
      structure_dict["Q0"] = "c"
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      structure_dict["Q"+str(j+1)] = "sc"

    elif (structure_case == 2):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = sc"
      structure_dict["Q"+str(j+1)] = "sc"

    elif (structure_case == 3):
      #structure_string_list.append("Q0 = sc"+str(j+1))
      structure_dict["Q0"] = "sc"+str(j+1)
      for i in range(k):
        #structure_string_list.append("Q"+str(i+1)+" = ss")
        structure_dict["Q"+str(i+1)] = "ss"
      #structure_string_list[j+1] = "Q"+str(j+1)+" = c"
      #structure_string_list[t+1] = "Q"+str(t+1)+" = c"
      structure_dict["Q"+str(j+1)] = "c"
      structure_dict["Q"+str(t+1)] = "c"

  return min_Delta_G, structure_dict

In [60]:
# Example to compute the minimal maximum-vertex-degree of CS2,2,2
determine_minimal_Delta_G_clique_star([2,2,2])

(3, {'Q0': 'c', 'Q1': 'ss', 'Q2': 'ss', 'Q3': 'ss'})

In [61]:
#for part in construct_list_of_multipartite_group_sizes(n=27,ni_min=5,k_min=3,k_max=7):
  #print(1+determine_minimal_Delta_G_clique_star(part)[0],determine_minimal_Delta_G_clique_star(part)[1],part)

In [66]:
def compare_minimal_edge_with_split_fuse_complete_multipartite(list_of_group_sizes):

  # Infer some parameters
  n = sum(list_of_group_sizes)
  k = len(list_of_group_sizes)
  split_fuse_count = n + (2*(k+1)) - 3

  decomposition_obj = minimal_edge_count_and_structure_complete_multipartite_orbit(list_of_group_sizes)
  min_edge = decomposition_obj[0]

  if (split_fuse_count < min_edge):
    print("Better split-fuse found! SF=",split_fuse_count,", min_edge=",min_edge)
    print(list_of_group_sizes)
    print(decomposition_obj[1])

  return None

In [67]:
# Example to compare a minimal edge representative locally equivalent to K8,8,8 with split-fuse.
compare_minimal_edge_with_split_fuse_complete_multipartite([8,8,8])

Better split-fuse found! SF= 29 , min_edge= 30
[8, 8, 8]
{'Q0': 'sc1', 'Q1': 'sc', 'Q2': 'ss', 'Q3': 'ss'}


In [34]:
def compare_minimal_edge_with_split_fuse_clique_star(list_of_group_sizes):

  # Infer some parameters
  n = sum(list_of_group_sizes)
  k = len(list_of_group_sizes)
  split_fuse_count = n + (2*(k+1)) - 3

  decomposition_obj = minimal_edge_count_and_structure_clique_star_orbit(list_of_group_sizes)
  min_edge = decomposition_obj[0]

  if (split_fuse_count < min_edge):
    print("Better split-fuse found! SF=",split_fuse_count,", min_edge=",min_edge)
    print(list_of_group_sizes)
    print(decomposition_obj[1])

  return None

In [62]:
#for part in construct_list_of_multipartite_group_sizes(27):
  #compare_minimal_edge_with_split_fuse_complete_multipartite(part)

In [63]:
#for part in construct_list_of_multipartite_group_sizes(27):
  #compare_minimal_edge_with_split_fuse_clique_star(part)

In [64]:
#construct_list_of_multipartite_group_sizes(n=24,ni_min=6,k_min=3,k_max=7)

In [73]:
def compute_split_fuse_parameters_for_complete_multipartite(list_of_group_sizes):

  # Infer base parameters
  n = sum(list_of_group_sizes)
  k = len(list_of_group_sizes)

  total_CZ = n + (2*(k+1)) - 3
  total_time_steps = 1 + max(k,max(list_of_group_sizes)+1)
  qubit_count = n + (2*(k+1)) - 2
  aux_qubits = qubit_count - n

  print("Group Sizes:",list_of_group_sizes)
  print("CZs: "+str(total_CZ)+
        "\ntime steps: "+str(total_time_steps)+
        "\nqubits: "+str(qubit_count)+
        " aux: "+str(aux_qubits)+"\n"
        )

  return None

In [76]:
# Example to compute the split-fuse resource requirments for preparing K8,8,8
compute_split_fuse_parameters_for_complete_multipartite([8,8,8])

Group Sizes: [8, 8, 8]
CZs: 29
time steps: 10
qubits: 30 aux: 6



In [77]:
#for part in construct_list_of_multipartite_group_sizes(n=20,ni_min=5,k_min=3,k_max=7):
  #compute_split_fuse_parameters_for_complete_multipartite(part)

In [42]:
# Function to write down a human-readable composition of LC transformation.
# Keeping with function-composition right-to-left notation, we reverse the list.
# Use the letter "o" in place of function composition symbol.

def write_pretty_LC_transformation(LC_class,LC_transformation_list):

  # Initialize an empty string to write down the transformation as a composition.
  LC_trans_string = ""
  if (LC_class == "CM"):
    print("LC transformation from Kn1,...,nk to graph:")
    for i in reversed(LC_transformation_list):
      LC_trans_string = LC_trans_string + "c"+str(i)+ " o "

  elif (LC_class == "CS"):
    print("LC transformation from CS^1n1,...,nk to graph:")
    for i in reversed(LC_transformation_list):
      LC_trans_string = LC_trans_string + "c"+str(i)+ " o "


  # Return the string, clipping the last three unneeded symbols.
  print(LC_trans_string[:-3])

In [43]:
from re import L
# Function to classify a graph based on its quotient graph decomposition.
# Also finds a transformation to/from the complete multipartite or clique-star.
# Requires the list of group sizes and a dictionary of their structure.
# For k groups of vertices n1,...,nk, there must be k+1 quotient graphs Q0,Q1,...,Qk.
# The dictionary of quotient graph structures must have keys "Q0", "Q1", etc.
# The values in these must be "c" or "sc1", "sc2," etc. for "Q0"
# For the remaining quotient graphs, they must be "c", "sc", or "ss".
# Assumes the a correct split decomposition (that all splits are strong).

# Input:
# list_of_group_sizes: the sizes of groups of vertices
# dict_of_quotient_graphs: the structure of each quotient graph

# Output:
# LC equivalence class representative (complete multipartite or clique-star)
# A transformation from the representative to the input graph
# A transformation from the input graph to the representative

def find_LC_transformation_for_graph(list_of_group_sizes,dict_of_quotient_graphs):

  # Infer some parameters
  n = sum(list_of_group_sizes)
  k = len(list_of_group_sizes)

  # Intialize a paramter to track where Q0 points, if not c.
  # Also initialize an empty string to determine the equivalence class.
  center_point_index = 0
  LC_class = ""

  if (len(dict_of_quotient_graphs) != (k+1)):
    print("ERROR: mismatched length of inputs, should be k+1 quotient graphs")
    return
  for (key,value) in dict_of_quotient_graphs.items():
    if (key == "Q0"):
      if ((value != "c") and (value[0:2] != "sc")):
        print("ERROR: Q0 must be c or sci")
        return
      elif (value[0:2] == "sc"):
        center_point_index = int(value[2:])
    else:
      if ((value != "c") and (value != "sc") and (value != "ss")):
        print("ERROR: Qi quotient graphs must be c, sc, or ss")
        return

    # Infer the indices of the vertices based on their groups.
    # Start from 1, and increase sequentially.
    dict_of_vertex_indices = dict()
    last_index = 0
    for (key,value) in dict_of_quotient_graphs.items():
      dict_of_vertex_indices[key] = []
      ni_index = int(key[1:])
      if (ni_index == 0):
        ni_size = 0
      else:
        ni_size = list_of_group_sizes[ni_index-1]
      for i in range(ni_size):
        last_index += 1
        dict_of_vertex_indices[key].append(last_index)

    # DEBUG
    #print("Dictionary of vertex indices")
    #print(dict_of_vertex_indices)

    # Count the number of quotient graphs which are ss
    ss_count = 0
    ss_index_set_I = set()
    for (key,value) in dict_of_quotient_graphs.items():
      if (value == "ss"):
        ss_count += 1
        ss_index_set_I.add(int(key[1:]))

    # DEBUG
    #print(ss_count, ss_index_set_I)

    # Initialize a list to track the LC transformations.
    # This is a list of functions to compose, starting from the first entry.
    LC_transformation_list = []

    # Break into cases based on the structure of Q0,Q1,...,Qk
    if (dict_of_quotient_graphs["Q0"] == "c"):
      # Case 1: Q0 is c
      if (len(ss_index_set_I)%2 == 0):
        # Even length implies complete multipartite case
        print("Graph is LC equivalent to complete multipartite.")
        LC_class = "CM"
        temp_I = ss_index_set_I.copy()
        while (len(temp_I) > 0):
          index1 = temp_I.pop()
          index2 = temp_I.pop()
          v1 = dict_of_vertex_indices["Q"+str(index1)][0]
          v2 = dict_of_vertex_indices["Q"+str(index2)][0]
          edge_pivot = [v1,v2,v1]
          LC_transformation_list = LC_transformation_list + edge_pivot

      else:
        # Odd length implies clique-star case
        print("Graph is LC equivalent to clique-star.")
        LC_class = "CS"
        temp_I = ss_index_set_I.copy()

        # By default, the clique-center has index 1
        # Remove this from the set of indices (if it's there)
        clique_center_index = 1
        clique_center_v = dict_of_vertex_indices["Q"+str(clique_center_index)][0]
        temp_I.remove(clique_center_index)
        while (len(temp_I) > 0):
          index1 = temp_I.pop()
          v1 = dict_of_vertex_indices["Q"+str(index1)][0]
          LC_transformation_list.append(v1)
        # Force the last transformation to use the center index.
        LC_transformation_list.append(clique_center_v)

    elif (dict_of_quotient_graphs["Q0"][0:2] == "sc"):

      # Infer a vertex in the center and remove it from I
      center_index = int(dict_of_quotient_graphs["Q0"][2:])
      center_type = dict_of_quotient_graphs["Q"+str(center_index)]
      center_v = dict_of_vertex_indices["Q"+str(center_index)][0]

      # DEBUG
      #print(center_index,center_type)

      if (center_type == "sc"):
        # Case 2: Qj is sc
        if (len(ss_index_set_I)%2 == 0):
          # Even length implies complete multipartite case
          print("Graph is LC equivalent to complete multipartite.")
          LC_class = "CM"
          LC_transformation_list.append(center_v)
          temp_I = ss_index_set_I.copy()
          while (len(temp_I) > 0):
            index1 = temp_I.pop()
            v1 = dict_of_vertex_indices["Q"+str(index1)][0]
            LC_transformation_list.append(v1)

        else:
          # Odd length implies clique-star case
          print("Graph is LC equivalent to clique-star.")
          LC_class = "CS"
          # If j != 1, use an edge pivot to shift the clique-center.
          if (center_index != 1):
            v1 = dict_of_vertex_indices["Q1"][0]
            v2 = dict_of_vertex_indices["Q"+str(center_index)][0]
            edge_pivot = [v1,v2,v1]
            LC_transformation_list = LC_transformation_list + edge_pivot

          temp_I = ss_index_set_I.copy()
          while (len(temp_I) > 0):
            index1 = temp_I.pop()
            v1 = dict_of_vertex_indices["Q"+str(index1)][0]
            LC_transformation_list.append(v1)


      elif (center_type == "c"):
        # Case 3: Qj is c
        if (len(ss_index_set_I)%2 == 1):
          # Odd length implies multipartite case
          print("Graph is LC equivalent to complete multipartite.")
          LC_class = "CM"
          LC_transformation_list.append(center_v)
          temp_I = ss_index_set_I.copy()
          while (len(temp_I) > 0):
            index1 = temp_I.pop()
            v1 = dict_of_vertex_indices["Q"+str(index1)][0]
            LC_transformation_list.append(v1)

        else:
          # Even length implies clique-star case
          print("Graph is LC equivalent to clique-star.")
          LC_class = "CS"
          # If j != 1, use an edge pivot to shift the clique-center.
          if (center_index != 1):
            v1 = dict_of_vertex_indices["Q1"][0]
            v2 = dict_of_vertex_indices["Q"+str(center_index)][0]
            edge_pivot = [v1,v2,v1]
            LC_transformation_list = LC_transformation_list + edge_pivot

          temp_I = ss_index_set_I.copy()
          while (len(temp_I) > 0):
            index1 = temp_I.pop()
            v1 = dict_of_vertex_indices["Q"+str(index1)][0]
            LC_transformation_list.append(v1)

    # Write down the LC transformation in a human readable format
    write_pretty_LC_transformation(LC_class,LC_transformation_list)

    return LC_class, LC_transformation_list

In [44]:
test_dict = {"Q0":"sc2","Q1":"ss","Q2":"c","Q3":"ss","Q4":"ss"}
find_LC_transformation_for_graph([2,2,2,2],test_dict)

Graph is LC equivalent to complete multipartite.
LC transformation from Kn1,...,nk to graph:
c7 o c5 o c1 o c3


('CM', [3, 1, 5, 7])

In [45]:
test_dict = {"Q0":"c","Q1":"ss","Q2":"ss","Q3":"ss"}
find_LC_transformation_for_graph([2,2,2],test_dict)

Graph is LC equivalent to clique-star.
LC transformation from CS^1n1,...,nk to graph:
c1 o c5 o c3


('CS', [3, 5, 1])

**Summary of useful functions**

Most functions use as input a list "list_of_group_sizes" = $[n_1,\cdots,n_k]$ corresponding to the sizes of the groups in a graph locally equivalent to a complete multipartite graph or a clique star.
Some functions compute or require as input the QASST decomposition of a graph, written as a Python dictionary "dict_of_quotient_graphs" with keys of the form "Q0", "Q1", etc., and values of the form "c", "sc", "ss", (complete, star-center, star-spoke), and possibly "scj" for "Q0".

1.   Compute the size of LC orbit.

*   $K_{n_1,\cdots,n_k}$: evaluate_sum_of_even_products_formula(list_of_group_sizes)
*   $CS^r_{n_1,\cdots,n_k}$: evaluate_sum_of_odd_products_formula(list_of_group_sizes)

2.   Compute the number of edges in a minimal-edge representative of the LC orbit, and give the QASST decomposition of a graph achieving this minimum.

*   $K_{n_1,\cdots,n_k}$: minimal_edge_count_and_structure_complete_multipartite_orbit(list_of_group_sizes)
*   $CS^r_{n_1,\cdots,n_k}$: minimal_edge_count_and_structure_clique_star_orbit(list_of_group_sizes)

3.   Compute the maximum vertex degree $\Delta(G)$ of a minimal-$\Delta(G)$ representative of the LC orbit, and give the QASST decomposition of a graph achieving this minimum.

*   $K_{n_1,\cdots,n_k}$: determine_minimal_Delta_G_complete_multipartite(list_of_group_sizes)
*   $CS^r_{n_1,\cdots,n_k}$ determine_minimal_Delta_G_clique_star(list_of_group_sizes)

4. Compute the resource requirements using the split-fuse method for a graph QASST-equivalent to a complete multipartite graph (it is the same formula for a clique-star). This includes the number of CZ gates, the number of time steps, and the number of qubits (including auxiliary qubits). The function returns no output, but writes a print-statement with these parameters.

*   compute_split_fuse_parameters_for_complete_multipartite(list_of_group_sizes)

5.   Function to identify an explicit LC transformation from an input graph state to either $K_{n_1,\cdots,n_k}$ or $CS^1_{n_1,\cdots,n_k}$, depending on which one the graph is locally equivalent to. The equivalence need not be specified (it is determined automatically by the function by the input QASST decomposition), but it assumes that this QASST decomposition is correct (i.e. consists of only valid strong splits with the appropriate graph-labeled tree structure).
The output specifies whether locally equivalent to complete multipartie (CM) or a clique-star (CS), and gives a list of the vertex indices required to transform $K_{n_1,\cdots,n_k}$ or $CS^1_{n_1,\cdots,n_k}$ into the input graph state using LC operations.

*   find_LC_transformation_for_graph(list_of_group_sizes,dict_of_quotient_graphs)


6.   Function to write down a composition sequence of LC transformations in a more human readable form such as $c_5\circ c_1\circ c_3$. This follows the usual function composition ordering, reading right to left. Requires that the orbit type of complete multipartite (CM) or clique-star (CS) be specified and an input list of vertex indices for the LC transformations. This function is called by the "find_LC_transformation_for_graph()" function by default.

*   write_pretty_LC_transformation(LC_class,LC_transformation_list)


These should be the functions of primary interest to a user wishing to experiment, other functions included are used as subroutines in these.
Below this summary, we include a working example of each of these functions which may be freely modified.

In [78]:
# Compute the size of the LC orbit for complete multipartite K2,2,2
evaluate_sum_of_even_products_formula([2,2,2])

40

In [80]:
# Compute the size of the LC orbit for clique-star CS2,2,2
evaluate_sum_of_odd_products_formula([2,2,2])

41

In [81]:
# Compute minimal edges and a representative for K2,2,2
minimal_edge_count_and_structure_complete_multipartite_orbit([2,2,2])

(6, {'Q0': 'sc1', 'Q1': 'sc', 'Q2': 'ss', 'Q3': 'ss'})

In [82]:
# Compute minimal edges and a representative for CS2,2,2
minimal_edge_count_and_structure_clique_star_orbit([2,2,2])

(6.0, {'Q0': 'c', 'Q1': 'ss', 'Q2': 'ss', 'Q3': 'ss'})

In [83]:
# Compute the minimal maximum-vertex-degree and a representative for K2,2,2
determine_minimal_Delta_G_complete_multipartite([2,2,2])

(3, {'Q0': 'sc1', 'Q1': 'sc', 'Q2': 'ss', 'Q3': 'ss'})

In [84]:
# Compute the minimal maximum-vertex-degree and a representative for CS2,2,2
determine_minimal_Delta_G_clique_star([2,2,2])

(3, {'Q0': 'c', 'Q1': 'ss', 'Q2': 'ss', 'Q3': 'ss'})

In [85]:
# Compute the split-fuse resource requirements for implementing K2,2,2
compute_split_fuse_parameters_for_complete_multipartite([2,2,2])

Group Sizes: [2, 2, 2]
CZs: 11
time steps: 4
qubits: 12 aux: 6



In [86]:
# Find an explicit LC transformation from CS2,2,2 to the minimal edge representative
find_LC_transformation_for_graph([2,2,2],{'Q0': 'c', 'Q1': 'ss', 'Q2': 'ss', 'Q3': 'ss'})

Graph is LC equivalent to clique-star.
LC transformation from CS^1n1,...,nk to graph:
c1 o c5 o c3


('CS', [3, 5, 1])